# 変分オートエンコーダ（VAE）Functional API編

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARIM-ACADEMY-2026/Advanced_Tutorial_3_Keras/blob/main/2_Keras_VAE_Functional-API.ipynb)

VAE（Variational Autoencoder）は、`3_Keras_AE_*-API.ipynb`のオートエンコーダ（AE）をさらに発展させた**生成モデル**（学習したデータに似た「新しいデータ」を作り出せるモデル）です。参考: [Keras公式のVAEサンプル](https://keras.io/examples/generative/vae/)。

## 対象読者・前提知識・動作環境・版とライセンス

- **対象読者**: `3_Keras_AE_Functional-API.ipynb`（オートエンコーダ・Functional API）を読んだ方
- **前提知識**: 「確率分布」「正規分布」という言葉に出会ったことがある程度でよい。VAE特有の数式（KLダイバージェンスなど）はこのノートブックの中で直感的な説明から始める
- **動作環境**: Python 3.10以降、TensorFlow 2.16以降・Keras 3系（`tensorflow.keras`は現在Keras 3を指す）。本シリーズで最も学習に時間がかかる
- **データセット**: MNIST（手書き数字、2〜4節）／Fashion-MNIST（5節）
- **版**: 2026-08-03作成

## オートエンコーダ（AE）とVAEは何が違うのか

前作のAEは、画像を「1つの点」（例えば2次元の座標）に圧縮していました。しかし、AEの潜在空間には保証がありません。学習に使われなかった、点と点の「間」の座標をデコーダに入力しても、意味のある画像になるとは限らない（歪んだ画像やノイズになりがち）のです。

VAEは、画像を「1つの点」ではなく「**確率分布**」（だいたいこのあたり、というぼんやりとした範囲）に圧縮するよう学習します。これにより、潜在空間全体が「なめらかに意味が変化する空間」になりやすく、学習データにない座標をデコーダに入力しても、それらしい画像を生成できるようになります。「圧縮・復元」が目的だったAEに対し、VAEは「学習データと似た、新しいデータを生成する」ことを目的とした生成モデルです。

## 目次

1. データの準備（MNIST）
2. モデルの構築（再パラメータ化トリック・KL損失）
3. 学習（MNIST）
4. 評価（潜在空間・生成画像）
5. Fashion-MNISTで学習し直し、まとめ

## 教材への接続（Google Colab）

Google Colabで開いた場合は、次のセルを実行してこのリポジトリをクローンし、`module/`（共通ヘルパー関数）や`output/`・`comparison_output/`（他ノートブックとの受け渡しファイル）を含むフォルダへ移動してください。ローカル環境でこのフォルダを直接開いている場合は、次のセルは実行不要です（すでにカレントディレクトリが正しい場所になっています）。


In [ ]:
!git clone https://github.com/ARIM-ACADEMY-2026/Advanced_Tutorial_3_Keras.git
%cd Advanced_Tutorial_3_Keras


## 1. データの準備

### ステップ1: ライブラリを読み込み、乱数シードを固定する

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers

BASE_DIR = Path.cwd()
sys.path.append(str(BASE_DIR))  # module/ はBASE_DIR直下にあるため、BASE_DIR自体をsys.pathに加える

from module.seed_utils import set_seed
from module import data_utils, viz_utils

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

set_seed(42)

### ステップ2: データセットを読み込む（訓練+テストを連結する）

これまでのMLP・CNN・AEでは「訓練データで学習し、テストデータで汎化性能を確認する」という枠組みでした。VAEのような生成モデルでは目的が異なり、「未知データへの当てはめの良さ」ではなく「データ全体の分布をどれだけ豊かに捉えられるか」が重要になります。そのため、Keras公式のVAEサンプルにならい、本ノートブックでも訓練データとテストデータを連結した全件（`all_digits`）を学習に使います。

一方で、潜在空間を可視化する際にはラベル（0〜9のどの数字か）で色分けしたいため、ラベル付きの訓練データ（`train_images`, `train_labels`）も別途保持しておきます。

In [ ]:
mnist_data = data_utils.load_digits_for_vae()

all_digits = mnist_data.all_digits
train_images_mnist = mnist_data.train_images
train_labels_mnist = mnist_data.train_labels

all_digits.shape, train_images_mnist.shape

**コラム: 「データリーケージ」に見えて、実は問題にならない理由**

コーディング規約では「訓練データだけで前処理をfitし、テストデータには漏らさない」という原則を重視しています。VAEで訓練・テストを連結して学習に使うのは、この原則に反しているように見えるかもしれません。しかし、ここで問題にしている「リーケージ」は、"教師あり学習でモデルの汎化性能を正しく見積もる"ための概念です。VAEはそもそも「未知データでの分類精度」のようなものを測る対象ではなく、生成モデルとしての性質（もっともらしい画像を生成できるか）を評価します。目的が変われば、守るべき手続きも変わるという例として捉えてください。

### ステップ3: 画像を確認する（図1）

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(train_images_mnist[i, :, :, 0], cmap="gray")
    plt.xlabel(str(train_labels_mnist[i]))
plt.suptitle("図1: MNISTの画像サンプル25枚")
plt.tight_layout()
plt.show()

## 2. モデルの構築

### ステップ1: エンコーダを組み立てる（画像 → 確率分布のパラメータ）

VAEのエンコーダは、AEのように「1つの潜在ベクトル」を直接出力するのではなく、「潜在空間上の正規分布のパラメータ」である**平均（`z_mean`）**と**分散の対数（`z_log_var`）**の2つを出力します。畳み込み層で画像から特徴を抽出する部分は、CNN編と似た構成です。

In [ ]:
from keras.layers import Input, Conv2D, Flatten, Dense

latent_dim = 2

encoder_inputs = Input(shape=(28, 28, 1))
c1 = Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
c2 = Conv2D(64, 3, activation="relu", strides=2, padding="same")(c1)
f3 = Flatten()(c2)
c4 = Dense(16, activation="relu")(f3)

z_mean = Dense(latent_dim, name="z_mean")(c4)
z_log_var = Dense(latent_dim, name="z_log_var")(c4)

z_mean.shape, z_log_var.shape

`Conv2D`に`strides=2`を指定すると、`MaxPooling2D`を別途使わなくても畳み込みと同時に画像を半分に縮小できます（28→14→7）。`z_mean`・`z_log_var`はどちらも`latent_dim=2`次元で、「潜在空間上のどのあたりに、どれくらいの広がりで存在するか」を表す2つのパラメータです。

**コラム: なぜ「分散」ではなく「分散の対数」を出力するのか**

分散は必ず0以上の値でなければなりませんが、`Dense`層の出力は正負どちらの値も取りえます。そこで「分散の対数（`log_var`）」を出力させ、使うときに`exp(log_var)`で指数変換すれば、常に正の値の分散が得られます。ニューラルネットワークの出力に「値の範囲の制約」がある場合、対数や`sigmoid`・`softmax`のような変換をはさんで制約を満たす、というのはよく使われるテクニックです。

### ステップ2: 再パラメータ化トリック（Reparameterization Trick）

エンコーダが出力した確率分布から、実際に1つの点`z`をサンプリング（抽出）する必要があります。しかし「確率分布からランダムに1点選ぶ」という操作は、そのままでは誤差逆伝播法（勾配を逆向きに伝えてパラメータを更新する仕組み）で扱えません。乱数を挟むと、その乱数がどのパラメータの影響でどう変化したかを微分できないためです。

そこで使われるのが**再パラメータ化トリック**です。「$z_{mean}$を中心に、$z_{log\_var}$の広がりでランダムに散らばった点」を、次のように書き換えます。

$$z = z_{mean} + \exp(0.5 \times z_{log\_var}) \times \epsilon$$

ここで$\epsilon$（イプシロン）は、標準正規分布（平均0・分散1）から取った乱数です。この式のポイントは、「乱数を使う部分（$\epsilon$）」と「学習で更新したいパラメータを使う部分（$z_{mean}$, $z_{log\_var}$）」を掛け算・足し算という微分可能な演算だけで分離していることです。$\epsilon$自体はモデルのパラメータに依存しない「ただの乱数」として扱われるため、$z_{mean}$・$z_{log\_var}$を生み出した層までは、通常どおり勾配を伝えることができます。

In [ ]:
class Sampling(layers.Layer):
    """再パラメータ化トリックにより、(z_mean, z_log_var) から潜在ベクトル z をサンプリングする層。"""

    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

encoder.summary()

`Sampling`は、独自の計算をする層をKerasで定義するときの基本形です。`keras.layers.Layer`を継承し、`call(self, inputs)`メソッドに実際の計算を書きます。この層には学習可能なパラメータ（重み）はなく、あくまで「決まった計算をする」ためだけの層です。

`encoder`は、1つの画像を入力すると`[z_mean, z_log_var, z]`という3つの出力を返すモデルです。「複数の出力を持つモデル」もFunctional APIならではの書き方です。

### ステップ3: デコーダを組み立てる（潜在ベクトル → 画像）

In [ ]:
from keras.layers import Reshape, Conv2DTranspose

latent_inputs = Input(shape=(latent_dim,), name="z_sampling")

d1 = Dense(7 * 7 * 64, activation="relu")(latent_inputs)
r1 = Reshape((7, 7, 64))(d1)
d2 = Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(r1)
d3 = Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(d2)
decoder_outputs = Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(d3)

decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

`Conv2DTranspose`は「転置畳み込み」と呼ばれる層で、`Conv2D`とは逆に画像を拡大します（7→14→28）。エンコーダで縮小した分だけ、デコーダで同じ倍率だけ拡大して元の28×28に戻す、対称的な設計になっています。

### ステップ4: VAE全体のモデルと、独自の損失関数を定義する

VAEの損失は2つの項の和です。

- **再構成損失（reconstruction loss）**: 元画像と、エンコード→デコードした画像がどれだけ近いか。AEと同じ考え方
- **KLダイバージェンス損失（KL loss）**: エンコーダが出力した確率分布（平均$z_{mean}$・分散$\exp(z_{log\_var})$の正規分布）が、標準正規分布（平均0・分散1）からどれだけ離れているかを測る指標。この項があることで、潜在空間の各点の周りに「無理やり標準正規分布に近い、ほどよい広がり」を持たせ、点と点の間にも意味のある空間が広がるよう促す

$$\text{total\_loss} = \text{reconstruction\_loss} + \text{kl\_loss}$$

2つの損失を同時に最小化するモデルは、Kerasの標準的な`compile(loss=...)`だけでは表現しにくいため、`train_step`メソッドを自分で書き換える「サブクラス化（Subclassing）API」を使います。このクラスは3節・5節どちらのデータセットで学習させるときも共通で使い回します。

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)
                )
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

`train_step`は、`model.fit()`が1バッチ処理するたびに内部で呼び出すメソッドです。通常は`compile(loss=...)`で指定した損失を使ってKerasが自動的にこの処理を行いますが、ここでは自分で書き換えることで、「2つの損失を合計してから勾配を計算する」という独自の学習手順を実現しています。`tf.GradientTape()`は「この中で行った計算を記録しておき、あとで微分（勾配計算）できるようにする」というTensorFlowの仕組みです。

**コラム: 3種類のKeras API、ここまでの使い分けまとめ**

- **Sequential API**（MLP編・AE編Sequential編）: 層を一本道に積むだけの、最もシンプルな書き方
- **Functional API**（CNN Functional編・AE Functional編・本ノートブックのencoder/decoder）: 層を関数のように呼び出し、分岐・合流・複数入出力を表現できる書き方
- **Subclassing API**（本ノートブックの`VAE`クラス）: `keras.Model`を継承し、`train_step`のような内部の挙動そのものを自分で書き換える、最も自由度が高いが難易度も高い書き方

「まずSequentialで基本を覚え、分岐が必要になったらFunctional、学習の手順自体を変えたくなったらSubclassing」という順序で必要に応じて使い分けるのが実務的です。

## 3. 学習（MNIST）

VAEは`compile()`時に`loss`を指定しません（`train_step`の中で自分で損失を計算しているため）。また、通常の`fit()`とは異なり`validation_data`は渡していません。これはSubclassing APIで`train_step`のみを定義し`test_step`を定義していないため、`validation_data`を指定すると別途評価用のステップが必要になるからです（生成モデルの評価は1節のコラムで触れたとおり、教師あり学習と同じ枠組みでは測れないため、ここでは訓練データ全体の学習に専念します）。

In [ ]:
vae_mnist = VAE(encoder, decoder)
vae_mnist.compile(optimizer="adam")

In [ ]:
%%time
history_mnist = vae_mnist.fit(
    all_digits,
    epochs=30,
    batch_size=256,
)

`vae_mnist = VAE(encoder, decoder)`は、既存の`encoder`・`decoder`（Functional APIの`Model`）を**そのまま**受け取ってラップしています。`vae_mnist.fit()`による学習は、内部で`encoder`・`decoder`が持つ層の重みを直接更新するため、学習後は`encoder`・`decoder`という変数自体が学習済みの状態になっています。

## 4. 評価（潜在空間・生成画像）

### ステップ1: 学習曲線を確認する（図2〜図4）

In [ ]:
viz_utils.plot_metric_curve(history_mnist, metric="loss", fig_num=2, output_dir=OUTPUT_DIR)

In [ ]:
viz_utils.plot_metric_curve(history_mnist, metric="reconstruction_loss", fig_num=3, output_dir=OUTPUT_DIR)

In [ ]:
viz_utils.plot_metric_curve(history_mnist, metric="kl_loss", fig_num=4, output_dir=OUTPUT_DIR)

`total_loss`（図2）は`reconstruction_loss`（図3、再構成の良さ）と`kl_loss`（図4、潜在空間の分布の整い方）の和です。学習が進むにつれて再構成損失は下がり続けやすい一方、KL損失はある程度のところで下げ止まる（0にはならない）ことがよくあります。KL損失を完全に0にする（＝標準正規分布に完全一致させる）と、潜在空間が画像ごとの違いを表現できなくなってしまうため、「ほどよいバランス」に落ち着くことが期待される設計です。なお、本ノートブックの`train_step`には検証（validation）の仕組みがないため、これらの図には検証データの曲線（赤線）がありません（MLP・CNN・AE編との違いに注意してください）。

### ステップ2: 潜在空間を可視化する（図5）

In [ ]:
z_mean_train, z_log_var_train, z_train = encoder.predict(train_images_mnist, verbose=0)

viz_utils.plot_latent_scatter(
    z_mean_train, train_labels_mnist,
    class_names=data_utils.MNIST_CLASS_NAMES, fig_num=5, output_dir=OUTPUT_DIR,
)

`encoder.predict()`は`[z_mean, z_log_var, z]`の3つを返すため、変数を3つ用意して受け取ります。可視化には、乱数によるサンプリングの影響を受けない`z_mean`（分布の中心）を使うのが一般的です。AE編の潜在空間と見比べると、VAEの方が同じ数字同士がより連続的な「かたまり」として、かつ全体が原点付近に集まった形で分布する傾向がよく見られます。これはKL損失が「原点を中心とした標準正規分布に近づける」よう働きかけているためです。

### ステップ3: 潜在空間から新しい画像を生成する（図6）

In [ ]:
def decode_fn(z_sample):
    return decoder.predict(z_sample, verbose=0)[0]


viz_utils.plot_latent_manifold(decode_fn, n=20, fig_num=6, output_dir=OUTPUT_DIR)

これがVAEの真骨頂です。潜在空間を格子状に区切り、**学習データにはなかった座標**をデコーダに入力して画像を生成しています。ある数字から別の数字へなめらかに形が変化していく様子が確認できれば、「潜在空間の間にも意味のある画像が存在する」というVAEの狙いが実現できていることになります（AEでは、このように格子状にサンプリングしても、意味のある画像になるとは限りません）。

### ステップ4: 元画像と再構成画像を見比べ、モデルを保存する（図7）

In [ ]:
reconstruction_train_vae = decoder.predict(z_mean_train[:10], verbose=0)

viz_utils.plot_reconstruction_grid(
    train_images_mnist[:10], reconstruction_train_vae,
    n=10, image_shape=(28, 28), fig_num=7, output_dir=OUTPUT_DIR,
)

In [ ]:
encoder.save(OUTPUT_DIR / "vae_encoder_mnist_trained.keras")
decoder.save(OUTPUT_DIR / "vae_decoder_mnist_trained.keras")
print("保存しました: vae_encoder_mnist_trained.keras / vae_decoder_mnist_trained.keras")

`VAE`クラスそのもの（`vae_mnist`）は保存しません。`Sampling`層や`VAE`クラスはPythonのクラス定義（コード）であり、`model.save()`で保存できる「重み」ではないためです。標準的な`Model`である`encoder`・`decoder`だけを保存すれば、必要な情報は失われません。

## 5. Fashion-MNISTで学習し直し、まとめ

最後に、2節〜4節と全く同じモデル構成を、**ゼロから**Fashion-MNISTで学習し直します。「数字」と「衣料品」という中身の異なるデータで、潜在空間の様子がどう変わるかを比較するのが目的です。

**編集メモ**: 底本ノートブックでは、MNISTで学習済みの`encoder`・`decoder`（重みが更新済みの状態）をそのまま使い回してFashion-MNISTを追加学習させていました（意図した実験というよりは、変数の使い回しによる結果と見られます）。「2つのデータセットでの潜在空間を独立に比較する」という本ノートブックの目的をはっきりさせるため、ここでは新しい`encoder`・`decoder`をゼロから構築し直しています。

### ステップ1: Fashion-MNISTを読み込む

In [ ]:
fashion_data = data_utils.load_fashion_mnist_for_vae()

all_images_fashion = fashion_data.all_images
train_images_fashion = fashion_data.train_images
train_labels_fashion = fashion_data.train_labels

all_images_fashion.shape, train_images_fashion.shape

### ステップ2: 2節と同じ構成でモデルをゼロから組み立てる

In [ ]:
encoder_inputs_fashion = Input(shape=(28, 28, 1))
c1f = Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs_fashion)
c2f = Conv2D(64, 3, activation="relu", strides=2, padding="same")(c1f)
f3f = Flatten()(c2f)
c4f = Dense(16, activation="relu")(f3f)
z_mean_fashion_layer = Dense(latent_dim, name="z_mean")(c4f)
z_log_var_fashion_layer = Dense(latent_dim, name="z_log_var")(c4f)
z_fashion_layer = Sampling()([z_mean_fashion_layer, z_log_var_fashion_layer])

encoder_fashion = keras.Model(
    encoder_inputs_fashion,
    [z_mean_fashion_layer, z_log_var_fashion_layer, z_fashion_layer],
    name="encoder_fashion",
)

latent_inputs_fashion = Input(shape=(latent_dim,), name="z_sampling_fashion")
d1f = Dense(7 * 7 * 64, activation="relu")(latent_inputs_fashion)
r1f = Reshape((7, 7, 64))(d1f)
d2f = Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(r1f)
d3f = Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(d2f)
decoder_outputs_fashion = Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(d3f)
decoder_fashion = keras.Model(latent_inputs_fashion, decoder_outputs_fashion, name="decoder_fashion")

encoder_fashion.summary()

再パラメータ化トリックやKLダイバージェンスの意味、`Sampling`層の役割は2節で説明したとおりです。ここでは`Sampling`クラスをそのまま再利用しつつ、`encoder_fashion`・`decoder_fashion`という新しい層・モデルをゼロから構築しています（`encoder`・`decoder`とは重みを一切共有していません）。

### ステップ3: VAEを学習する

In [ ]:
vae_fashion = VAE(encoder_fashion, decoder_fashion)
vae_fashion.compile(optimizer="rmsprop")

底本の教材にならい、オプティマイザは`"rmsprop"`（MNIST編は`"adam"`）を使います。オプティマイザの違いによる収束の速さ・安定性の違いも、余力があれば演習問題で比較してみてください。`VAE`クラスは2節で定義したものをそのまま再利用しています（同じ構造の学習に何度も使い回せるのが、クラスとして定義しておく利点です）。

In [ ]:
%%time
history_fashion = vae_fashion.fit(
    all_images_fashion,
    epochs=100,
    batch_size=64,
)

**コラム: MNIST編よりエポック数が多い理由**

Fashion-MNIST（衣料品）はMNIST（手書き数字）より画像内の模様やテクスチャが複雑で、同じ再構成品質に達するにはより多くの学習が必要になる傾向があります。「同じモデル構成でも、データセットの複雑さによって適切な学習量は変わりうる」という、実務でもよく直面する判断の一例です。

### ステップ4: 学習済みモデルを保存し、潜在空間を可視化する（図8・図9）

In [ ]:
encoder_fashion.save(OUTPUT_DIR / "vae_encoder_fashion_trained.keras")
decoder_fashion.save(OUTPUT_DIR / "vae_decoder_fashion_trained.keras")
print("保存しました: vae_encoder_fashion_trained.keras / vae_decoder_fashion_trained.keras")

In [ ]:
z_mean_fashion, z_log_var_fashion, z_fashion = encoder_fashion.predict(train_images_fashion, verbose=0)

viz_utils.plot_latent_scatter(
    z_mean_fashion, train_labels_fashion,
    class_names=data_utils.FASHION_MNIST_CLASS_NAMES, fig_num=8, output_dir=OUTPUT_DIR,
    filename="fig8_latent_scatter_fashion.png",
)

In [ ]:
def decode_fn_fashion(z_sample):
    return decoder_fashion.predict(z_sample, verbose=0)[0]


viz_utils.plot_latent_manifold(
    decode_fn_fashion, n=20, fig_num=9, output_dir=OUTPUT_DIR,
    filename="fig9_latent_manifold_fashion.png",
)

4節（MNIST）の潜在空間・生成画像と見比べてみてください。MNISTでは「数字の形」ごとにクラスタが分かれていましたが、Fashion-MNISTでは「上半身の衣類（T-shirt/Shirt/Pulloverなど）」と「履物（Sandal/Sneaker/Ankle boot）」のように、より大まかなシルエットの違いでクラスタが分かれる傾向がよく見られます。これは、VAEが「見た目（画素の並び方）の似ている物」を近づけるように学習しているためで、人間が付けた「T-shirtかShirtか」といったカテゴリの境界と必ずしも一致するとは限らない、という点は覚えておく価値があります。

### ステップ5: MNIST編との損失を比較する

In [ ]:
final_recon_mnist = history_mnist.history["reconstruction_loss"][-1]
final_kl_mnist = history_mnist.history["kl_loss"][-1]
final_recon_fashion = history_fashion.history["reconstruction_loss"][-1]
final_kl_fashion = history_fashion.history["kl_loss"][-1]

print(f"MNIST   : 最終reconstruction_loss={final_recon_mnist:.3f}, 最終kl_loss={final_kl_mnist:.3f}")
print(f"Fashion : 最終reconstruction_loss={final_recon_fashion:.3f}, 最終kl_loss={final_kl_fashion:.3f}")

再構成損失を単純比較するときは注意が必要です。Fashion-MNISTの方がMNISTより画像内の情報量（模様やグラデーション）が多いため、同程度のモデル・エポック数でも再構成損失は一般に大きくなりがちです。「損失の値そのもの」よりも、「学習曲線が寝てきているか（まだ改善の余地があるか）」「生成画像を目で見て破綻していないか」を合わせて確認することが大切です。

In [ ]:
import csv

comparison_path = OUTPUT_DIR / "vae_comparison.csv"
rows = [
    {"dataset": "MNIST", "final_reconstruction_loss": final_recon_mnist, "final_kl_loss": final_kl_mnist},
    {"dataset": "Fashion-MNIST", "final_reconstruction_loss": final_recon_fashion, "final_kl_loss": final_kl_fashion},
]
with open(comparison_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["dataset", "final_reconstruction_loss", "final_kl_loss"])
    writer.writeheader()
    writer.writerows(rows)

import pandas as pd
pd.read_csv(comparison_path)

## まとめ

- VAEは、画像を「1つの点」ではなく「確率分布（平均・分散）」に圧縮し、再パラメータ化トリックでその分布から1点をサンプリングして復元する生成モデルであることを学んだ
- 損失は「再構成損失」と「KLダイバージェンス損失」の和で、`train_step`をサブクラス化（Subclassing API）して自分で計算する必要があった
- 潜在空間を格子状にサンプリングしてデコードすることで、学習データにない新しい画像を生成できることを確認した（AEにはない、VAEならではの性質）
- MNIST（数字）とFashion-MNIST（衣料品）という異なるデータセットで同じ構成を学習し、潜在空間のクラスタの現れ方の違いを比較した

## 本ノートブックで扱っていないこと（今後の課題）

- 潜在空間の次元数（ここでは可視化のため2次元に固定）を増やした場合の生成品質の変化
- KL損失に重み（$\beta$）を掛けて再構成とのバランスを調整する$\beta$-VAE
- 条件付きVAE（Conditional VAE）のように、ラベル情報を使って「特定のクラスの画像を狙って生成する」拡張
- GAN（敵対的生成ネットワーク）など、VAE以外の生成モデルとの比較

## 演習問題

1. `latent_dim`を2から8に増やしたモデルを作り、再構成損失がどう改善するか確認しなさい（可視化にはPCAなどで別途2次元に落とす工夫が必要になる）。
2. KL損失の計算式に重み`beta`（例: `kl_loss = beta * tf.reduce_mean(...)`）を掛け、`beta`を0.5、1、5と変えたときに潜在空間の散布図（図5相当）・生成画像（図6相当）がどう変わるか観察しなさい。
3. 4節（MNIST）と5節（Fashion-MNIST）の潜在空間マニフォールド（図6・図9）を並べて表示し、それぞれどのような「軸」（例: z[0]方向に何が変化するか）になっているか、自分の言葉で説明しなさい。
4. 編集メモで触れた「学習済みの重みを引き継いで追加学習する」（底本の元のやり方）と「ゼロから学習し直す」（本ノートブックのやり方）を両方試し、生成画像の質やクラスタの現れ方にどのような違いが出るか比較しなさい。